# Fine-tune Laya for DevSecOps finding triage
Use the current project source and independently reviewed training data. Select Kaggle GPU T4 x2 when available. Keep the final test split outside this session. See docs/LAYA-TRAINING.md for the label rubric and evaluation procedure.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
PROJECT = Path(os.environ.get('DEVSECOPS_SOURCE', '/kaggle/working/DevSecOps-Agent'))
DATA = Path(os.environ.get('DEVSECOPS_DATA', '/kaggle/input/laya-triage-data'))
OUTPUT = Path('/kaggle/working/laya-devsecops-triage')
assert (PROJECT / 'pyproject.toml').is_file(), 'Unpack the current source distribution and set PROJECT to its directory.'
assert (DATA / 'manifest.json').is_file(), 'Set DATA to the reviewed dataset directory.'
assert not (DATA / 'test.jsonl').exists(), 'Retain the test split outside the training session.'


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', str(PROJECT) + '[training]'], check=True)
import torch
assert torch.cuda.is_available(), 'Enable a CUDA GPU notebook accelerator.'
GPU_COUNT = min(2, torch.cuda.device_count())
print({'gpus': GPU_COUNT, 'names': [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]})


In [ ]:
subprocess.run([sys.executable, '-m', 'vulnscan.training.finetune', '--data', str(DATA), '--mode', 'validate'], check=True)
manifest = json.loads((DATA / 'manifest.json').read_text())
print({'counts': manifest['counts'], 'routes': manifest['route_counts'], 'groups': manifest['independent_groups']})


## Train and calibrate
The run uses at most four epochs, FP16, 512 tokens, gradient checkpointing, microbatch 2 per GPU, accumulation 16, and seed 42. Validation selects the checkpoint; the separate calibration split fits temperatures. The script saves checkpoints locally.


In [ ]:
env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
command = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=' + str(GPU_COUNT), '-m', 'vulnscan.training.finetune', '--data', str(DATA), '--out', str(OUTPUT), '--mode', 'train', '--epochs', '4', '--microbatch', '2', '--accumulation', '16', '--seed', '42']
subprocess.run(command, env=env, check=True)


In [ ]:
import hashlib
checksums = json.loads((OUTPUT / 'checksums.json').read_text())
for relative, expected in checksums.items():
    digest = hashlib.sha256()
    with (OUTPUT / relative).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    assert digest.hexdigest() == expected, relative
config = json.loads((OUTPUT / 'rl_agent_config.json').read_text())
assert 'temperature_by_options' not in config
print({'checkpoint': str(OUTPUT), 'training': config['training'], 'temperatures': config['temperature']})


## Final test and rollout
Download the checkpoint files and evaluate locally against the retained test split with `python -m vulnscan.training.evaluate`. Compare the frozen base checkpoint and trained checkpoint. Inspect per-route recall, dangerous downgrades, human-review errors, calibration and latency. Keep advisory routing disabled until the acceptance criteria in docs/LAYA-TRAINING.md pass. No hub publication is performed by this notebook.
